# Time Series Forecasting Analysis

## Project Overview
This notebook demonstrates **time series forecasting** using three popular models:
- **ARIMA** (AutoRegressive Integrated Moving Average)
- **SARIMA** (Seasonal ARIMA)
- **Prophet** (Facebook's forecasting tool)

We'll analyze monthly airline passenger data and compare the performance of these models using interactive Plotly visualizations.

### What is Time Series Forecasting?
Time series forecasting predicts future values based on previously observed values over time. It's widely used in:
- Sales forecasting
- Stock price prediction
- Weather forecasting
- Demand planning

## 1. Import Required Libraries

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# For ARIMA and SARIMA
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# For Prophet
from prophet import Prophet

# For visualizations
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# For metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. Load and Explore Dataset
We'll use the classic **Airline Passengers** dataset which contains monthly totals of international airline passengers from 1949 to 1960.

In [20]:
# Load the dataset
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url)

# Parse dates
df['Month'] = pd.to_datetime(df['Month'])
df1 = df
df.set_index('Month', inplace=True)
df.columns = ['Passengers']

print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nLast few rows:")
print(df.tail())
print("\nDataset Info:")
print(df.info())
print("\nBasic Statistics:")
print(df.describe())

Dataset Shape: (144, 1)

First few rows:
            Passengers
Month                 
1949-01-01         112
1949-02-01         118
1949-03-01         132
1949-04-01         129
1949-05-01         121

Last few rows:
            Passengers
Month                 
1960-08-01         606
1960-09-01         508
1960-10-01         461
1960-11-01         390
1960-12-01         432

Dataset Info:
<class 'pandas.DataFrame'>
DatetimeIndex: 144 entries, 1949-01-01 to 1960-12-01
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Passengers  144 non-null    int64
dtypes: int64(1)
memory usage: 2.2 KB
None

Basic Statistics:
       Passengers
count  144.000000
mean   280.298611
std    119.966317
min    104.000000
25%    180.000000
50%    265.500000
75%    360.500000
max    622.000000


## 3. Visualize Time Series Data
Let's create an interactive visualization to explore the data pattern.

In [4]:
# Create interactive plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Passengers'],
    mode='lines',
    name='Monthly Passengers',
    line=dict(color='royalblue', width=2)
))

fig.update_layout(
    title='Airline Passengers Over Time (1949-1960)',
    xaxis_title='Date',
    yaxis_title='Number of Passengers',
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig.show()

In [26]:
X = df1['Month']
y = df1['Passengers']
from sklearn.preprocessing import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.head()) 
print(y_train.head())


KeyError: 'Month'

### Visualization Insights:
The plot reveals three key patterns:
1. **Trend**: Clear upward trend over time - passenger numbers consistently increase year over year
2. **Seasonality**: Regular repeating patterns within each year - peaks and valleys occur at similar times annually
3. **Variance**: The magnitude of seasonal fluctuations increases with the trend (multiplicative seasonality)

These patterns suggest we need models that can handle both trend and seasonality.

## 4. Prepare Data for Modeling
We'll split the data into training (80%) and testing (20%) sets.

In [5]:
# Split data into train and test sets
train_size = int(len(df) * 0.8)
train = df[:train_size]
test = df[train_size:]

print(f"Training set size: {len(train)} months")
print(f"Testing set size: {len(test)} months")
print(f"\nTraining period: {train.index[0]} to {train.index[-1]}")
print(f"Testing period: {test.index[0].strftime('%Y-%m')} to {test.index[-1].strftime('%Y-%m')}")

Training set size: 115 months
Testing set size: 29 months

Training period: 1949-01-01 00:00:00 to 1958-07-01 00:00:00
Testing period: 1958-08 to 1960-12


## 5. ARIMA Model Implementation
**ARIMA** (p, d, q) stands for:
- **p**: Number of autoregressive terms
- **d**: Number of differences needed for stationarity
- **q**: Number of moving average terms

We'll use ARIMA(1,1,1) for simplicity.

In [6]:
# Build and fit ARIMA model
# p = 1 → use 1 past value pattern
# d = 1 → difference once
# q = 1 → use 1 past error
arima_model = ARIMA(train['Passengers'], order=(1, 1, 1))
arima_fitted = arima_model.fit()

# Make predictions on test set
arima_predictions = arima_fitted.forecast(steps=len(test))

print("ARIMA Model Summary:")
print(arima_fitted.summary())
print("\n" + "="*50)
print(f"First 5 predictions: {arima_predictions[:5].values}")

ARIMA Model Summary:
                               SARIMAX Results                                
Dep. Variable:             Passengers   No. Observations:                  115
Model:                 ARIMA(1, 1, 1)   Log Likelihood                -526.123
Date:                Thu, 18 Jun 2026   AIC                           1058.246
Time:                        13:26:22   BIC                           1066.454
Sample:                    01-01-1949   HQIC                          1061.577
                         - 07-01-1958                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.5111      0.114     -4.488      0.000      -0.734      -0.288
ma.L1          0.9144      0.056     16.251      0.000       0.804       1.025
sigma2       592.7851    101.20

## 6. Visualize ARIMA Results

In [7]:
# Visualize ARIMA predictions
fig = go.Figure()

# Training data
fig.add_trace(go.Scatter(
    x=train.index,
    y=train['Passengers'],
    mode='lines',
    name='Training Data',
    line=dict(color='blue', width=2)
))

# Actual test data
fig.add_trace(go.Scatter(
    x=test.index,
    y=test['Passengers'],
    mode='lines',
    name='Actual Test Data',
    line=dict(color='green', width=2)
))

# ARIMA predictions
fig.add_trace(go.Scatter(
    x=test.index,
    y=arima_predictions,
    mode='lines',
    name='ARIMA Predictions',
    line=dict(color='red', width=2, dash='dash')
))

fig.update_layout(
    title='ARIMA Model: Actual vs Predicted',
    xaxis_title='Date',
    yaxis_title='Number of Passengers',
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig.show()

### ARIMA Results Analysis:
**Observations:**
- ARIMA captures the **overall upward trend** in passenger numbers
- However, it **struggles with seasonality** - the predictions are relatively smooth without capturing the peaks and valleys
- The model underestimates actual values during peak seasons and overestimates during low seasons
- **Limitation**: Standard ARIMA doesn't explicitly model seasonal patterns, which is why we see this gap

This highlights the need for seasonal models like SARIMA!

## 7. SARIMA Model Implementation
**SARIMA** extends ARIMA by adding seasonal components: (p,d,q)(P,D,Q,s)
- **(P,D,Q)**: Seasonal AR, differencing, and MA terms
- **s**: Length of seasonal cycle (12 for monthly data with yearly seasonality)

We'll use SARIMA(1,1,1)(1,1,1,12).

In [8]:
# Build and fit SARIMA model
sarima_model = SARIMAX(train['Passengers'], 
                        order=(1, 1, 1), 
                        seasonal_order=(1, 1, 1, 12))
sarima_fitted = sarima_model.fit(disp=False)

# Make predictions
sarima_predictions = sarima_fitted.forecast(steps=len(test))

print("SARIMA Model Summary:")
print(sarima_fitted.summary())
print("\n" + "="*50)
print(f"First 5 predictions: {sarima_predictions[:5].values}")

SARIMA Model Summary:
                                     SARIMAX Results                                      
Dep. Variable:                         Passengers   No. Observations:                  115
Model:             SARIMAX(1, 1, 1)x(1, 1, 1, 12)   Log Likelihood                -374.130
Date:                            Thu, 18 Jun 2026   AIC                            758.260
Time:                                    13:26:34   BIC                            771.385
Sample:                                01-01-1949   HQIC                           763.575
                                     - 07-01-1958                                         
Covariance Type:                              opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.6366      0.334     -1.908      0.056      -1.290       0.017
ma.L1        

## 8. Visualize SARIMA Results

In [9]:
# Visualize SARIMA predictions
fig = go.Figure()

# Training data
fig.add_trace(go.Scatter(
    x=train.index,
    y=train['Passengers'],
    mode='lines',
    name='Training Data',
    line=dict(color='blue', width=2)
))

# Actual test data
fig.add_trace(go.Scatter(
    x=test.index,
    y=test['Passengers'],
    mode='lines',
    name='Actual Test Data',
    line=dict(color='green', width=2)
))

# SARIMA predictions
fig.add_trace(go.Scatter(
    x=test.index,
    y=sarima_predictions,
    mode='lines',
    name='SARIMA Predictions',
    line=dict(color='orange', width=2, dash='dash')
))

fig.update_layout(
    title='SARIMA Model: Actual vs Predicted',
    xaxis_title='Date',
    yaxis_title='Number of Passengers',
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig.show()

### SARIMA Results Analysis:
**Significant Improvement!**
- SARIMA **captures both trend and seasonality** effectively
- The predictions now show the **characteristic peaks and valleys** matching the seasonal pattern
- Much better alignment with actual test data compared to ARIMA
- The seasonal component (12-month cycle) allows the model to learn and replicate yearly patterns
- **Why it works better**: The seasonal order (1,1,1,12) explicitly models the repeating 12-month patterns we saw in the data

SARIMA is clearly superior to ARIMA for this seasonal dataset!

## 9. Prophet Model Implementation
**Prophet** is Facebook's time series forecasting tool designed to handle:
- Strong seasonal patterns
- Holiday effects
- Missing data
- Trend changes

Prophet requires data in specific format with columns: `ds` (date) and `y` (value).

In [10]:
# Prepare data for Prophet
train_prophet = train.reset_index()
train_prophet.columns = ['ds', 'y']

# Build and fit Prophet model
prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
prophet_model.fit(train_prophet)

# Create future dataframe for predictions
future = prophet_model.make_future_dataframe(periods=len(test), freq='MS')
prophet_forecast = prophet_model.predict(future)

# Extract predictions for test period
prophet_predictions = prophet_forecast.iloc[-len(test):]['yhat'].values

print("Prophet model trained successfully!")
print(f"\nFirst 5 predictions: {prophet_predictions[:5]}")
print(f"\nForecast columns available: {list(prophet_forecast.columns)}")

13:26:43 - cmdstanpy - INFO - Chain [1] start processing
13:26:45 - cmdstanpy - INFO - Chain [1] done processing


Prophet model trained successfully!

First 5 predictions: [448.05867604 418.19149519 387.55089988 361.35661017 388.23804391]

Forecast columns available: ['ds', 'trend', 'yhat_lower', 'yhat_upper', 'trend_lower', 'trend_upper', 'additive_terms', 'additive_terms_lower', 'additive_terms_upper', 'yearly', 'yearly_lower', 'yearly_upper', 'multiplicative_terms', 'multiplicative_terms_lower', 'multiplicative_terms_upper', 'yhat']


## 10. Visualize Prophet Results

In [11]:
# Visualize Prophet predictions with uncertainty intervals
fig = go.Figure()

# Training data
fig.add_trace(go.Scatter(
    x=train.index,
    y=train['Passengers'],
    mode='lines',
    name='Training Data',
    line=dict(color='blue', width=2)
))

# Actual test data
fig.add_trace(go.Scatter(
    x=test.index,
    y=test['Passengers'],
    mode='lines',
    name='Actual Test Data',
    line=dict(color='green', width=2)
))

# Prophet predictions
fig.add_trace(go.Scatter(
    x=test.index,
    y=prophet_predictions,
    mode='lines',
    name='Prophet Predictions',
    line=dict(color='purple', width=2, dash='dash')
))

# Add uncertainty interval
test_forecast = prophet_forecast.iloc[-len(test):]
fig.add_trace(go.Scatter(
    x=test.index,
    y=test_forecast['yhat_upper'].values,
    mode='lines',
    name='Upper Bound',
    line=dict(width=0),
    showlegend=False
))

fig.add_trace(go.Scatter(
    x=test.index,
    y=test_forecast['yhat_lower'].values,
    mode='lines',
    name='Confidence Interval',
    line=dict(width=0),
    fillcolor='rgba(128, 0, 128, 0.2)',
    fill='tonexty'
))

fig.update_layout(
    title='Prophet Model: Actual vs Predicted (with Uncertainty)',
    xaxis_title='Date',
    yaxis_title='Number of Passengers',
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig.show()

In [12]:
# Visualize Prophet components (trend and seasonality)
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Trend Component', 'Yearly Seasonal Component'),
    vertical_spacing=0.15
)

# Trend
fig.add_trace(
    go.Scatter(x=prophet_forecast['ds'], y=prophet_forecast['trend'],
               mode='lines', name='Trend', line=dict(color='blue', width=2)),
    row=1, col=1
)

# Yearly seasonality
fig.add_trace(
    go.Scatter(x=prophet_forecast['ds'], y=prophet_forecast['yearly'],
               mode='lines', name='Yearly Seasonality', line=dict(color='green', width=2)),
    row=2, col=1
)

fig.update_xaxes(title_text="Date", row=1, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)
fig.update_yaxes(title_text="Trend", row=1, col=1)
fig.update_yaxes(title_text="Seasonal Effect", row=2, col=1)

fig.update_layout(
    title='Prophet Model Components Decomposition',
    height=700,
    showlegend=True,
    template='plotly_white'
)

fig.show()

### Prophet Results Analysis:
**Key Insights:**

**First Visualization (Predictions):**
- Prophet captures seasonality very well, similar to SARIMA
- The **shaded confidence interval** shows prediction uncertainty - wider intervals indicate less confidence
- Predictions align closely with actual values, showing good model fit
- Unlike ARIMA/SARIMA, Prophet provides built-in uncertainty quantification

**Second Visualization (Components):**
1. **Trend Component**: Shows the smooth, long-term upward trajectory - passenger numbers consistently growing
2. **Yearly Seasonal Component**: Reveals the repeating annual pattern
   - Peaks around **July-August** (summer travel season)
   - Troughs around **February** (winter low season)
   - This decomposition helps understand **what drives** the predictions

**Prophet's Advantage**: Automatic decomposition into interpretable components makes it easier to understand and explain the forecast!

## 11. Compare Model Performance
Let's evaluate all three models using standard metrics:
- **MAE** (Mean Absolute Error): Average absolute difference
- **RMSE** (Root Mean Squared Error): Penalizes larger errors more
- **MAPE** (Mean Absolute Percentage Error): Error as a percentage

In [13]:
# Calculate metrics for all models
def calculate_metrics(actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    return mae, rmse, mape

# Calculate for each model
arima_mae, arima_rmse, arima_mape = calculate_metrics(test['Passengers'], arima_predictions)
sarima_mae, sarima_rmse, sarima_mape = calculate_metrics(test['Passengers'], sarima_predictions)
prophet_mae, prophet_rmse, prophet_mape = calculate_metrics(test['Passengers'], prophet_predictions)

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Model': ['ARIMA', 'SARIMA', 'Prophet'],
    'MAE': [arima_mae, sarima_mae, prophet_mae],
    'RMSE': [arima_rmse, sarima_rmse, prophet_rmse],
    'MAPE (%)': [arima_mape, sarima_mape, prophet_mape]
})

print("Model Performance Comparison:")
print("="*60)
print(comparison_df.to_string(index=False))
print("="*60)
print("\n💡 Lower values indicate better performance")

Model Performance Comparison:
  Model       MAE      RMSE  MAPE (%)
  ARIMA 85.254156 97.499620 21.291349
 SARIMA 23.555558 30.141832  5.052683
Prophet 33.896315 41.332489  7.719629

💡 Lower values indicate better performance


In [14]:
# Visualize all predictions together
fig = go.Figure()

# Actual test data (thicker line for emphasis)
fig.add_trace(go.Scatter(
    x=test.index,
    y=test['Passengers'],
    mode='lines+markers',
    name='Actual',
    line=dict(color='black', width=3),
    marker=dict(size=6)
))

# ARIMA predictions
fig.add_trace(go.Scatter(
    x=test.index,
    y=arima_predictions,
    mode='lines',
    name=f'ARIMA (MAE: {arima_mae:.2f})',
    line=dict(color='red', width=2, dash='dash')
))

# SARIMA predictions
fig.add_trace(go.Scatter(
    x=test.index,
    y=sarima_predictions,
    mode='lines',
    name=f'SARIMA (MAE: {sarima_mae:.2f})',
    line=dict(color='orange', width=2, dash='dot')
))

# Prophet predictions
fig.add_trace(go.Scatter(
    x=test.index,
    y=prophet_predictions,
    mode='lines',
    name=f'Prophet (MAE: {prophet_mae:.2f})',
    line=dict(color='purple', width=2, dash='dashdot')
))

fig.update_layout(
    title='Model Comparison: All Predictions vs Actual',
    xaxis_title='Date',
    yaxis_title='Number of Passengers',
    hovermode='x unified',
    template='plotly_white',
    height=500,
    legend=dict(x=0.02, y=0.98)
)

fig.show()

In [15]:
# Create bar chart for metrics comparison
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Mean Absolute Error (MAE)', 'Root Mean Squared Error (RMSE)', 'Mean Absolute % Error (MAPE)'),
    horizontal_spacing=0.12
)

models = comparison_df['Model']
colors = ['red', 'orange', 'purple']

# MAE
fig.add_trace(
    go.Bar(x=models, y=comparison_df['MAE'], marker_color=colors, name='MAE', showlegend=False),
    row=1, col=1
)

# RMSE
fig.add_trace(
    go.Bar(x=models, y=comparison_df['RMSE'], marker_color=colors, name='RMSE', showlegend=False),
    row=1, col=2
)

# MAPE
fig.add_trace(
    go.Bar(x=models, y=comparison_df['MAPE (%)'], marker_color=colors, name='MAPE', showlegend=False),
    row=1, col=3
)

fig.update_yaxes(title_text="MAE", row=1, col=1)
fig.update_yaxes(title_text="RMSE", row=1, col=2)
fig.update_yaxes(title_text="MAPE (%)", row=1, col=3)

fig.update_layout(
    title='Performance Metrics Comparison (Lower is Better)',
    height=400,
    template='plotly_white'
)

fig.show()

### Final Model Comparison Analysis:

**Visualization Insights:**

1. **Combined Predictions Plot:**
   - **ARIMA** (red dashed): Fails to capture seasonality, smooth but inaccurate
   - **SARIMA** (orange dotted): Closely follows actual pattern with seasonal peaks/valleys
   - **Prophet** (purple dash-dot): Also captures seasonality well, very similar to SARIMA

2. **Metrics Bar Charts:**
   - **Clear Winner**: Either SARIMA or Prophet (very close performance)
   - **Worst Performer**: ARIMA (significantly higher errors across all metrics)
   - The bars show ARIMA has roughly **2-3x higher errors** than the seasonal models

---

## 🎯 Conclusions:

| Model | Best For | Limitations |
|-------|----------|-------------|
| **ARIMA** | Non-seasonal data, simple trends | Cannot handle seasonality well |
| **SARIMA** | Seasonal data, statistical rigor | Requires parameter tuning, computationally intensive |
| **Prophet** | Business forecasting, interpretability | May need holiday/event data for best results |

### **Recommendation for this dataset:**
Use **SARIMA or Prophet** - both capture the seasonal patterns effectively. Choose based on:
- **SARIMA**: When you need statistical significance testing
- **Prophet**: When you need easy interpretation and uncertainty quantification

**Key Takeaway**: For seasonal time series, always use models that explicitly handle seasonality (SARIMA, Prophet) rather than basic ARIMA!